# msg_parts

> The canonical LLM message model - `Msg` and the `Part` classes - and its formatted text form

In [ ]:
#| default_exp msg_parts

`aidialog.msg_parts` holds the data structures every layer of the ecosystem shares: the canonical in-memory form of an LLM conversation (`Msg` and the `Part` subclasses), the helpers that build them from plain content (`mk_msg`, `mk_msgs`), and the fenced-JSON text form that serializes tool calls and usage into markdown (`fmt2hist`, `hist2fmt`). It depends only on `fastcore`, so packages that need to speak the message format - dialog libraries, transcript tools, compaction - get it without depending on an LLM client. `fastllm` builds its chat clients on top of these same types.

In [ ]:
#| export
import base64, json, copy
from json import dumps
from fastcore.utils import *
from fastcore.xtras import detect_mime

In [ ]:
#| hide
from fastcore.test import *
from fastcore.xml import Safe
from IPython.display import Markdown

## Part

Canonical atomic content unit for multimodal inputs/outputs.

**Why it exists**
- Providers represent content blocks differently (`input_text`, `text`, `inlineData`, `source`, etc).
- `Part` gives one stable internal shape so serialization/parsing logic can stay at provider boundaries.

**Design Notes**
- `Part` is a base class: each kind of content is a subclass with its own fields, registered under the wire tag it serializes as (`text`, `input_image`, ...), which stays readable as `.type`.
- Behaviour that varies by kind - how a part displays while streaming, how it renders into the formatted text form - is a method override, so nothing dispatches on `.type`.
- `raw` carries the vendor's original block on any part, for round-trips that need details we don't model (an Anthropic thinking signature, say).

**Connections**
- Wrapped by `Msg.content`.
- Produced/consumed by `highlevel` coercion, provider serializers in `clients`, and normalizers in `normalize`.
- Key enabler for model-only swapping without rewriting message construction code.

All four providers represent content parts differently in their wire format:

- **[OpenAI Responses](https://developers.openai.com/api/reference/resources/responses/methods/create)** — Content is an array of typed parts: `{"type": "input_text", "text": "..."}`, `{"type": "input_image", "image_url": "..."}`, `{"type": "input_audio", "input_audio": {"data": "...", "format": "..."}}`, `{"type": "input_file", "file_data": "data:application/pdf;base64,...", "filename": "..."}`
- **[OpenAI-compat (Chat)](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)** — Content is an array of typed parts: `{"type": "text", "text": "..."}`, `{"type": "image_url", "image_url": {"url": "..."}}`, `{"type": "input_audio", "input_audio": {"data": "...", "format": "..."}}`, `{"type": "file", "file": {"file_data": "data:application/pdf;base64,...", "filename": "..."}}`
- **[Anthropic](https://docs.anthropic.com/en/api/messages)** — Content blocks with `source` nesting: `{"type": "text", "text": "..."}`, `{"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": "..."}}`, `{"type": "document", "source": {"type": "base64", "media_type": "application/pdf", "data": "..."}}`
- **[Gemini](https://ai.google.dev/api/generate-content)** — A `parts` array of `{text}` or `{inlineData: {mimeType, data}}` or `{fileData: {mimeType, fileUri}}`: `{"text": "..."}`, `{"inlineData": {"mimeType": "image/jpeg", "data": "..."}}`, `{"fileData": {"mimeType": "application/pdf", "fileUri": "..."}}`

> **Spec sources:** OpenAI Responses `Input{Text,Image,File}Content` (`specs/openai.with-code-samples.yml:68093–68444`), OpenAI Chat `ChatCompletionRequestMessageContentPart*` (`specs/openai.with-code-samples.yml:35185–35321`), Anthropic `Request{Text,Image,Document}Block` (`specs/anthropic.yml:12159–12458`), Gemini `Part`/`Blob`/`FileData` (`specs/gemini.json:189–387`).

`Part` canonicalizes these into a uniform `(type, text, data)` triple. The canonical types, their aliases, and expected input formats:

| Canonical `Part.type` | Aliases accepted | Expected input format | `Part.text` | `Part.data` |
|---|---|---|---|---|
| `"text"` | — | Plain string | `"the text"` | `None` |
| `"input_image"` | `image`, `image_url` | URL, data URL (`data:image/...;base64,...`), or provider-native `source`/`inlineData` in `data` | URL shorthand | `{"url": "..."}` or `{"image_url": "..."}` or provider-native payload |
| `"input_audio"` | `audio` | base64 data + format (OpenAI), URL/`fileUri` (Gemini). Not supported on Anthropic. | URL shorthand | `{"input_audio": {"data": "...", "format": "wav"}}` or `{"inlineData": {...}}` |
| `"input_video"` | `video`, `video_url` | URL or `fileUri`. Currently Gemini only; OpenAI maps to `input_file`. Not supported on Anthropic. | URL shorthand | `{"video_url": "..."}` or `{"fileData": {...}}` |
| `"input_file"` | `file`, `pdf`, `document` | data URL, URL, `file_id`, or provider-native `source` in `data` | URL shorthand | `{"file_data": "data:...;base64,...", "filename": "..."}` or `{"source": {...}}` |

**Provider support matrix:**

| Canonical type | OpenAI Responses | OpenAI-compat (Chat) | Anthropic | Gemini |
|---|---|---|---|---|
| `text` | ✅ `input_text` | ✅ `text` | ✅ `text` | ✅ `text` |
| `input_image` | ✅ `input_image` | ✅ `image_url` | ✅ `image` + `source` | ✅ `inlineData` / `fileData` |
| `input_audio` | ✅ `input_audio` | ✅ `input_audio` | ❌ (raises `UnsupportedCapabilityError`) | ✅ `inlineData` / `fileData` |
| `input_video` | ✅ (mapped to `input_file`) | ✅ (mapped to `input_file`) | ❌ (raises `UnsupportedCapabilityError`) | ✅ `fileData` |
| `input_file` | ✅ `input_file` | ✅ `file` | ✅ `document` + `source` | ✅ `inlineData` / `fileData` |

**Escape hatch:** For any type, setting `Part.data["<provider>"]` (e.g. `Part.data["anthropic"]`) to a dict bypasses canonicalization and passes the payload directly to that provider (see `_provider_part`).

**Canonicalization design:** `Part.data` uses OpenAI-style key conventions as the single canonical input format. Users write one shape (e.g. `{"image_url": "..."}`, `{"file_data": "data:...;base64,..."}`, `{"input_audio": {"data": "...", "format": "wav"}}`), and the provider serializers (`_openai_responses_part`, `_anthropic_part`, `_gemini_part`) dispatch these into provider-native wire formats. This means users can swap models/providers without changing their `Part` construction code. `Part.text` serves as a URL shorthand fallback — e.g. `Part(type="input_image", text="https://example.com/img.png")` works when you just have a URL and no other metadata.

The key insight: `type` carries the **semantic modality**, not the provider-specific type name. Provider-specific payload details live in `data`, keeping `Part` itself provider-agnostic. This means downstream code can branch on `Part.type` without knowing which provider produced it.

In [ ]:
#| export
class Part(BasicRepr):
    "Base class for content parts; subclasses register under the wire tag they serialize as."
    text = None    # subclasses that carry text declare it as an attribute; the rest read as None
    reg = {}       # wire tag -> subclass
    def __init__(self,
        *,
        raw=None,          # The vendor's original block, kept for lossless round-trips
        cache_control=None # Prompt-cache directive for providers that take one
    ):
        store_attr()
    def __init_subclass__(cls, tag=None, **kw):
        super().__init_subclass__(**kw)
        if tag: cls.type,Part.reg[tag] = tag,cls
    def __eq__(self, o): return type(o) is type(self) and self.__dict__ == o.__dict__
    def __hash__(self): return hash((type(self).__name__, self.text))
    def replace(self, **kw):
        "A copy of this part with `kw` attributes changed"
        res = copy.copy(self)
        res.__dict__.update(kw)
        return res

In [ ]:
#| export
PartType = str_enum('PartType', 'text', 'thinking', 'refusal', 'tool_use', 'server_tool_result', 'tool_result',
                    'input_image', 'input_audio', 'input_video', 'input_file')

`PartType` is the wire vocabulary: the tag each kind of content serializes as, and the key each subclass registers under. The content kinds are these; tool calls and their results are `Part`s too, defined with their own section below.

In [ ]:
#| export
class Text(Part, tag=PartType.text):
    "Plain text content."
    def __init__(self, text=None, citations=None, **kw):
        super().__init__(**kw)
        store_attr('text,citations')

class Thinking(Part, tag=PartType.thinking):
    "A reasoning block; `showthink` asks renderers to show the thought rather than a 🧠 glyph."
    def __init__(self, text=None, showthink=False, **kw):
        super().__init__(**kw)
        store_attr('text,showthink')

class Refusal(Part, tag=PartType.refusal):
    "A provider's refusal to answer."
    def __init__(self, text=None, **kw):
        super().__init__(**kw)
        store_attr('text')

class Media(Part):
    "Media content: `text` is a URL or data URL."
    def __init__(self, text=None, mime=None, **kw):
        super().__init__(**kw)
        store_attr('text,mime')

class InputImage(Media, tag=PartType.input_image): "An image input."
class InputAudio(Media, tag=PartType.input_audio): "An audio input."
class InputVideo(Media, tag=PartType.input_video): "A video input."
class InputFile (Media, tag=PartType.input_file ): "A file input."

def mk_part(type, **kw):
    "The `Part` subclass registered for wire tag `type`, built from `kw`"
    return Part.reg[type](**kw)

In [ ]:
#| export
def _trunc_strs(o, n=200):
    "Truncate str or dict"
    if not o: return o
    if isinstance(o,str) and len(o)>n: return o[:100]+'...'
    if isinstance(o,dict): return {k: (v[:100]+'...' if isinstance(v,str) and len(v)>n else v) for k,v in o.items()}
    return o

@patch
def _repr_markdown_(self:Part):
    flds = '\n'.join(f"- {k}: `{_trunc_strs(getattr(self,k))}`" for k in self.__stored_args__ if k not in ('text','cache_control'))
    return f"""**{type(self).__name__}** (`{self.type}`)

{_trunc_strs(self.text) if self.text else ''}

::: details

{flds}

:::"""

In [ ]:
Text('Hello world!'*30, raw={'long':"10"*150})

**Text** (`text`)

Hello world!Hello world!Hello world!Hello world!Hello world!Hello world!Hello world!Hello world!Hell...

::: details

- raw: `{'long': '1010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010...'}`
- citations: `None`

:::

## Msg

Canonical conversation turn abstraction.

**Why it exists**
- Conversation structures vary across APIs (chat messages, response input items, content blocks).
- `Msg` lets the rest of fastllm reason in one conversation format.

**Design Notes**
- `role` captures turn semantics (`user`, `assistant`, `tool`, etc).
- `content` is a list of `Part` to support multimodal turns consistently.

**Connections**
- Primary input to `acompletion` and provider clients.
- Used by toolloop replay flows (`StreamSummary`/`Completion` -> `Msg` coercion in `highlevel`).

All three providers structure conversation messages differently:

- **[OpenAI Chat](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)** — Messages are `{role, content}` objects. Roles: `system`, `developer`, `user`, `assistant`, `tool`. Content is a string or array of typed parts. Tool results use `role: "tool"` with `tool_call_id`.
- **[OpenAI Responses](https://developers.openai.com/api/reference/resources/responses/methods/create)** — Input items with `{role, content}`. Text parts become `input_text`/`output_text` depending on role. Tool results are `{type: "function_call_output", call_id, output}`.
- **[Anthropic](https://docs.anthropic.com/en/api/messages)** — Messages are `{role, content}` with roles `user` or `assistant` only. System prompt is a separate top-level parameter. Tool results are `tool_result` content blocks inside `role: "user"` messages.
- **[Gemini](https://ai.google.dev/api/generate-content)** — Messages are `{role, parts}`. Roles: `user` or `model`. System prompt uses `system_instruction`. Tool results are `functionResponse` parts inside `role: "user"` messages.

> **Spec sources:** OpenAI `ChatCompletionRequest*Message` (`specs/openai.with-code-samples.yml:35022–35444`), Anthropic `InputMessage` + `RequestToolResultBlock` (`specs/anthropic.yml:11140–11159,12595–12652`), Gemini `Content` (`specs/gemini.json:171–188`).

`Msg` normalizes these into a canonical `(role, content, data)` triple:

| Canonical `Msg.role` | OpenAI Chat | OpenAI Responses | Anthropic | Gemini |
|---|---|---|---|---|
| `"system"` | `role: "system"` | `role: "system"` | Separate `system` param | `system_instruction` |
| `"user"` | `role: "user"` | `role: "user"` | `role: "user"` | `role: "user"` |
| `"assistant"` | `role: "assistant"` | `role: "assistant"` | `role: "assistant"` | `role: "model"` |
| `"tool"` | `role: "tool"` + `tool_call_id` | `function_call_output` + `call_id` | `role: "user"` + `tool_result` block | `role: "user"` + `functionResponse` |

**`Msg.data` metadata:** Provider-agnostic metadata that doesn't fit `role`/`content`:
- `tool_calls` — list of tool call dicts (assistant messages)
- `tool_call_id` / `call_id` / `id` — tool call identifier (tool result messages)
- `name` — tool function name (tool result messages)
- `is_error` — whether the tool result is an error (Anthropic-specific, forwarded)

**Escape hatch:** Like `Part`, setting `Msg.data["<provider>"]` (e.g. `Msg.data["anthropic"]`) to a dict with `"role"` bypasses serialization entirely and passes the raw message to that provider.

In [ ]:
#| export
class Msg(BasicRepr):
    "A normalized message."
    def __init__(self,
        role,   # 'user', 'assistant', or 'tool'
        content # list of `Part`
    ):
        store_attr()

    @property
    def text(self): return ''.join(p.text or '' for p in self.content if isinstance(p, Text))

    def __eq__(self, o): return type(o) is type(self) and self.__dict__ == o.__dict__
    def __hash__(self): return hash((self.role, len(self.content)))

    def _repr_markdown_(self):
        return f"""**Msg**

- role: `{self.role}`

<contents>

{'\n\n'.join(p._repr_markdown_() for p in self.content)}

</contents>"""

In [ ]:
Msg('user', content=[Text('Hello world!', raw={'long':"10"*150})]*3)

**Msg**

- role: `user`

<contents>

**Text** (`text`)

Hello world!

::: details

- raw: `{'long': '1010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010...'}`
- citations: `None`

:::

**Text** (`text`)

Hello world!

::: details

- raw: `{'long': '1010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010...'}`
- citations: `None`

:::

**Text** (`text`)

Hello world!

::: details

- raw: `{'long': '1010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010...'}`
- citations: `None`

:::

</contents>

## Tool parts

A tool call and its result are content parts like any other: they arrive interleaved with text in an assistant message, and they render into the formatted text form alongside it. All four providers represent tool invocations differently in their wire format:

- **[OpenAI Responses](https://developers.openai.com/api/reference/resources/responses/methods/create)** — Tool calls are flat output items: `{type: "function_call", call_id, name, arguments}` where `arguments` is a JSON **string**. Streamed via `response.function_call_arguments.delta` events.
- **[OpenAI-compat (Chat)](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)** — Tool calls are nested: `{id, type: "function", function: {name, arguments}}` where `arguments` is a JSON **string**. Streamed in chunks via `tool_calls[i].function.arguments` deltas.
- **[Anthropic](https://docs.anthropic.com/en/api/messages)** — Tool calls are `tool_use` content blocks: `{id, name, input, type: "tool_use"}` where `input` is a parsed JSON object. Streamed via `input_json_delta` events.
- **[Gemini](https://ai.google.dev/api/generate-content)** — Tool calls are `functionCall` parts: `{id, name, args}` where `args` is a parsed JSON object. Not chunked during streaming.

> **Spec sources:** OpenAI Responses `FunctionToolCall` (`specs/openai.with-code-samples.yml:44347–44389`), OpenAI Chat `ChatCompletionMessageToolCall` (`specs/openai.with-code-samples.yml:34869–34894`), Anthropic `ResponseToolUseBlock` (`specs/anthropic.yml:13563–13588`), Gemini `FunctionCall` (`specs/gemini.json:273–293`).

`ToolUse` canonicalizes these into a flat `(id, name, arguments)` triple where `arguments` is always a parsed `dict`:

| Field | OpenAI Responses | OpenAI-compat (Chat) | Anthropic | Gemini | `ToolUse` |
|---|---|---|---|---|---|
| ID | `call_id` | `id` | `id` | `id` | `id` |
| Name | `name` | `function.name` | `name` | `name` | `name` |
| Args | `arguments` (JSON string) | `function.arguments` (JSON string) | `input` (object) | `args` (object) | `arguments` (dict) |

`ToolResult` carries the same four identifying fields plus the result `text`, so a call and its result share one shape and one renderer.

In [ ]:
#| export
class _ToolPart(Part):
    "Shared shape of a tool call and its result."
    def __init__(self, id=None, name=None, arguments=None, server=False, text=None, **kw):
        if arguments is None: arguments = {}
        super().__init__(**kw)
        store_attr('id,name,arguments,server,text')

class ToolUse  (_ToolPart, tag=PartType.tool_use   ): "A tool invocation; `server` marks one the provider ran itself."
class ToolResult(_ToolPart, tag=PartType.tool_result): "A tool call's result, `text` holding the output."
class ServerToolResult(Part, tag=PartType.server_tool_result):
    "A provider-side tool result, kept as `raw` for round-trips."
    def __init__(self, text=None, **kw):
        super().__init__(**kw)
        store_attr('text')

In [ ]:
#| export
@patch
def _repr_markdown_(self:ToolUse):
    return f"""🔧 **{self.name}**(`{self.arguments}`)

::: details

- id: `{self.id}`
- server: `{self.server}`
- raw: `{_trunc_strs(self.raw)}`

:::"""

def display_list(l): 
    from IPython.display import Markdown, display
    display(Markdown('\n\n'.join(o._repr_markdown_() for o in l)))

In [ ]:
ToolUse(id='oxwvx1fm', name='simple_add', arguments={'b': 547982745, 'a': 5478954793}, raw={'thoughtSignature': 'EscDCsQDAQw51scPHdv+D5BX7JWdLzz3Bv8tsKFRuAJe2UkTFZ+NZKzNsLtmQBiia+/r4HJEUptq1zQB0q9HToX0qzCUqyNAbDLY76KxMeW9jpsnUvh6ZjPM5sDD7fAafF7cjdApNMsihPqIZBAZjAlFPcp1c/50MObH5f1q7hO7fgDS4iSJ3Q3FfbAYWnJ4nlA2peVMu/6WFcKZh1wcZCIuN6iFCj6nhH+6RKkaFRaM0b6XCmpti6qldSeZx+qtHmo+lzr1tct4Gz/CITDI7gRJ3qfLYV2u45jOhKzdd1t6gQ39XLJ93j0xd0AwpzcdZLbHWqwWJCQ43nNzhJ7IQTAWOSyPgKDnlAMHq2PTEoXBYkMBApCZ1x+HncBzt77kQrTTe7sWGVmD5boVnYAIFPFGXOULP5tDZ+nog+Fg8NV10vaFKlHVf+VDzFnVWxT259LN12ykGtBilfpTXiKCV12RAZwhuL7vXXHrsBGg5HNVImcXqgMvwf/rtQlJeop+9bEcAiU48hMFMzumOrCmmHD3HgxpYLW7T3vtDmbNdKCDqVtIwO4Rp5HE6GudRWmq8iC2UnyQglUXoXVnxIZW7eYYDsGAYrYgZ1A='})

🔧 **simple_add**(`{'b': 547982745, 'a': 5478954793}`)

::: details

- id: `oxwvx1fm`
- server: `False`
- raw: `{'thoughtSignature': 'EscDCsQDAQw51scPHdv+D5BX7JWdLzz3Bv8tsKFRuAJe2UkTFZ+NZKzNsLtmQBiia+/r4HJEUptq1zQB0q9HToX0qzCUqyNAbDLY...'}`

:::

## Completion

All four providers return non-stream responses in different shapes. `Completion` normalizes these into a canonical `(model, message, finish_reason, usage, raw)` object.

**Provider response → `Completion` field mapping:**

| Field | OpenAI Responses | OpenAI-compat (Chat) | Anthropic | Gemini |
|---|---|---|---|---|
| `model` | `raw.model` | `raw.model` | `raw.model` | passthrough (input model) |
| `message` | `output[].content[]` → `Msg(role="assistant", content=[Part...])` | `choices[0].message.content` → `Msg` | `content[]` → `Msg` | `candidates[0].content.parts[]` → text joined into single `Part` |
| `finish_reason` | `raw.status` (e.g. `"completed"`) | `choices[0].finish_reason` (e.g. `"stop"`) | `raw.stop_reason` (e.g. `"end_turn"`) | `candidates[0].finishReason` (e.g. `"STOP"`) |
| `usage` | `usage_from_openai(raw.usage)` | `usage_from_openai(raw.usage)` | `usage_from_anthropic(raw.usage)` | `usage_from_gemini(raw.usageMetadata)` |
| `tool_calls` | `output[]` where `type=="function_call"` → `ToolUse` | `message.tool_calls[].function` → `ToolUse` | `content[]` where `type=="tool_use"` → `ToolUse` | `parts[]` with `functionCall` → `ToolUse` |
| `raw` | full response dict | full response dict | full response dict | full response dict |

**Notable differences:**
- Tool calls live in `message.content` as `ToolUse` parts; `Completion.tool_calls` is a view over them, so the two can never disagree
- **Gemini** joins all text parts into a single string rather than preserving individual parts
- **Gemini** uses the input `model` string directly since the response only contains `modelVersion` (a version identifier, not a model name)

In [ ]:
#| export
class Completion(BasicRepr):
    "Normalized completion response."
    def __init__(self, model, message, finish_reason=None, usage=None, api_name=None, vendor_name=None, raw=None):
        if raw is None: raw = {}
        store_attr()
    def __eq__(self, o): return type(o) is type(self) and self.__dict__ == o.__dict__
    def __hash__(self): return hash((self.model, self.finish_reason, self.api_name, self.vendor_name))

    @property
    def tool_calls(self):
        "The `ToolUse` parts of `message`: a call lives in the content, not beside it"
        return [p for p in self.message.content if isinstance(p, ToolUse)]

In [ ]:
comp = Completion('claude-sonnet-4-20250514', Msg('assistant', [Text('The sum is 8. '), ToolUse(id='t1', name='add', arguments={'a':3,'b':5})]), finish_reason='tool_use')
test_eq(len(comp.tool_calls), 1)
comp.tool_calls[0]

🔧 **add**(`{'a': 3, 'b': 5}`)

::: details

- id: `t1`
- server: `False`
- raw: `None`

:::

## Message utilities

`mk_tool_res_msg` pairs parallel tool calls with their results as one `tool`-role message, one `tool_result` part per call.

In [ ]:
#| export
def mk_tool_res_msg(tool_calls:list[ToolUse], results:list[str|list]):
    'A util to prepare parallel tool call with str or media list results'
    parts = [ToolResult(id=tc.id, name=tc.name, arguments=tc.arguments, server=tc.server, text=res)
             for tc,res in zip(tool_calls, results)]
    return Msg(role="tool", content=parts)

`sys_text` and `part_txt` accept either bare strings or `Part`s, so call sites need not care which form they hold.

In [ ]:
#| export
def sys_text(system):
    "Extract text from system (str or Part)."
    if system is None: return None
    return system if isinstance(system, str) else system.text

def part_txt(p): return p.text if isinstance(p,Part) else p

Media referenced by URL needs a MIME type. `data_url` parses `data:` URLs, and `url_mime` guesses from the extension, falling back to fetching the resource: its Content-Type header decides when specific, and its first bytes are sniffed when the header is generic (octet-stream, text/plain) or missing; the fetch imports `httpx` lazily, so aidialog's hard dependencies stay at fastcore alone. `MediaUrl` bundles a URL with its MIME type - the reference form for media passed by URL rather than downloaded.


In [ ]:
#| export
@flexicache(time_policy(24*3600))
def _fetch_url_partial(url, nbytes=512):
    "Fetch remote media, returning `(content_type_header, bytes)`; bytes optionally only the first `nbytes`."
    import httpx  # deliberately lazy: keeps aidialog's deps to fastcore alone (may re-base on fastcore.net later)
    try:
        with httpx.stream('GET', url, headers={'Range': f'bytes=0-{nbytes-1}'}, follow_redirects=True) as r:
            if r.status_code not in (200, 206): return None, None
            return r.headers.get('content-type'), r.read()
    except (httpx.HTTPError, httpx.InvalidURL): return None, None

In [ ]:
#| export
_ext_mime = {
    '.jpg':'image/jpeg', '.jpeg':'image/jpeg', '.png':'image/png', '.gif':'image/gif', '.webp':'image/webp', '.svg':'image/svg+xml',
    '.pdf':'application/pdf',
    '.mp3':'audio/mpeg', '.wav':'audio/wav', '.ogg':'audio/ogg', '.flac':'audio/flac', '.m4a':'audio/mp4',
    '.mp4':'video/mp4', '.mov':'video/quicktime', '.webm':'video/webm',
}

def data_url(url):
    "Parse data:mime;base64,data URL into (mime, b64_data), or None."
    if not isinstance(url, str) or not url.startswith('data:') or ',' not in url: return None
    header, body = url.split(',', 1)
    if ';base64' not in header or not body: return None
    return header[5:].split(';',1)[0].strip() or 'application/octet-stream', body

def url_mime(url, default='application/octet-stream'):
    "Guess mime from URL extension, then the server's Content-Type, then sniffed bytes; never None."
    if "youtube.com" in url or "youtu.be" in url: return "video/mp4"
    ext = '.' + url.rsplit('.', 1)[-1].split('?')[0].lower() if '.' in url.split('?')[0].split('/')[-1] else ''
    if (mime:=_ext_mime.get(ext)) is None:
        ctype, data = _fetch_url_partial(url)
        mime = (ctype or '').split(';')[0].strip().lower()
        if not mime or mime in ('application/octet-stream', 'text/plain'): mime = detect_mime(data)  # generic headers say nothing
    return ifnone(mime, default)

In [ ]:
test_eq(url_mime('https://example.com/badge.svg'), 'image/svg+xml')
test_eq(url_mime('nonsense.xyz123'), 'application/octet-stream')   # sniff path dead-ends: `default` applies, never None

In [ ]:
_real = _fetch_url_partial
def _fetch_url_partial(url, nbytes=512): return 'image/svg+xml; charset=utf-8', b'<svg xmlns="http://www.w3.org/2000/svg"/>'
test_eq(url_mime('https://example.com/chart'), 'image/svg+xml')   # specific Content-Type wins; params stripped
def _fetch_url_partial(url, nbytes=512): return 'application/octet-stream', b'\x89PNG\r\n\x1a\n' + b'\0'*16
test_eq(url_mime('https://example.com/pic'), 'image/png')         # generic Content-Type: the bytes decide
def _fetch_url_partial(url, nbytes=512): return 'text/plain', b'some text'
test_eq(url_mime('https://example.com/thing'), 'application/octet-stream')   # generic header, unsniffable bytes: default
_fetch_url_partial = _real

In [ ]:
#| export
class MediaUrl(BasicRepr):
    "Direct URL media reference"
    def __init__(self, url, mime=None): self.url, self.mime = url, ifnone(mime, url_mime(url))

Content values become `Part`s through one dispatch: a str is text, bytes are sniffed with `detect_mime` and inlined as a base64 data URL, and a `MediaUrl` becomes a reference part. fastllm's `mk_msg` builds on this dispatch, adding provider concerns (`Completion` unwrapping, cache control) on its side of the boundary.

In [ ]:
#| export
def _mime2part_cls(mime):
    "The `Media` subclass for MIME string `mime`"
    if mime.startswith('image/'): return InputImage
    if mime.startswith('audio/'): return InputAudio
    if mime.startswith('video/'): return InputVideo
    return InputFile

def _bytes2content(data):
    "Convert bytes to fastllm canonical content"
    mtype = detect_mime(data)
    if not mtype: raise ValueError(f'Data must be a supported file type, got {data[:10]}')
    encoded = base64.b64encode(data).decode("utf-8")
    return _mime2part_cls(mtype)(f'data:{mtype};base64,{encoded}', mime=mtype)

def _url2content(o):
    "Convert MediaUrl to fastllm canonical content"
    mime = o.mime or url_mime(o.url)
    return _mime2part_cls(mime)(o.url, mime=mime)

def mk_content(o):
    "Convert a content value (str, bytes, `MediaUrl`, or already a `Part`) to a canonical `Part`"
    if isinstance(o, str):        return Text(o)
    elif isinstance(o, bytes):    return _bytes2content(o)
    elif isinstance(o, MediaUrl): return _url2content(o)
    return o

## The formatted text form

A conversation with tool calls can be rendered as one markdown string, with each call and its result serialized as a fenced JSON block whose info string is `json {.tool}` (and token usage as `json {.usage}`). This fixture is in the *legacy* envelope format that earlier fastllm releases shipped; `conv_tools` upgrades it to the fenced wire format, and is idempotent:

In [ ]:
fmt_outp = '''
I'll solve this step-by-step, using parallel calls where possible.

<details class='tool-usage-details' markdown='1'>

```json
{
  "id": "toolu_01KjnQH2Nsz2viQ7XYpLW3Ta",
  "call": { "function": "simple_add", "arguments": { "a": 10, "b": 5 } },
  "result": "15",
  "server": false
}
```

</details>

<details class='tool-usage-details' markdown='1'>

```json
{
  "id": "toolu_01Koi2EZrGZsBbnQ13wuuvzY",
  "call": { "function": "simple_add", "arguments": { "a": 2, "b": 1 } },
  "result": "3",
  "server": false
}
```

</details>

Now I need to multiply 15 * 3 before I can do the final division:

<details class='tool-usage-details' markdown='1'>

```json
{
  "id": "toolu_0141NRaWUjmGtwxZjWkyiq6C",
  "call": { "function": "multiply", "arguments": { "a": 15, "b": 3 } },
  "result": "45",
  "server": false
}
```

</details>

<details class='token-usage-details' markdown='1'><summary>Cache hit: 81.8% | Tokens: total=23,276 input=23,158 (+18,910 cached, 0 new) output=118 (reasoning 23)</summary>

`Usage(prompt_tokens=3, completion_tokens=10, total_tokens=13, raw={'input_tokens': 3, 'cache_creation_input_tokens': 2079, 'cache_read_input_tokens': 2070, 'cache_creation': {'ephemeral_5m_input_tokens': 2079, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 10, 'service_tier': 'standard', 'inference_geo': 'global'})`

</details>
'''

In [ ]:
#| export
tool_info = 'json {.tool}'     # fence info string of a tool block: {id, name, args, result} (+server; `error` reserved)
usage_info = 'json {.usage}'   # fence info string of a usage block: UsageStats fields

def parse_tools(s):
    "Split `s` into `(text, data)` segments: `data` is a parsed `{.tool}` block dict, `None` for the final segment"
    res, pos = [], 0
    for info,body,start,end in fenced_blocks(s):
        if info != tool_info: continue
        try: d = json.loads(body)
        except Exception: continue
        res.append((s[pos:start], d))
        pos = end
    return res + [(s[pos:], None)]

def strip_tools(s, tools=True, usage=True):
    "Remove `{.tool}` (and `{.usage}`) blocks from `s`"
    out, pos = [], 0
    for info,body,start,end in fenced_blocks(s):
        if not (tools and info == tool_info) and not (usage and info == usage_info): continue
        out.append(s[pos:start])
        pos = end
    out.append(s[pos:])
    return ''.join(out)

think_start,think_end = '<!--think_start-->','<!--think_end-->'
re_think = re.compile(rf'{re.escape(think_start)}.*?{re.escape(think_end)}\n?', re.DOTALL)

# Frozen legacy envelope recognition, used only by `conv_tools`. These are the
# exact patterns fastllm shipped for the released `<details markdown='1'>`
# envelopes plus the never-released `::: {.details}` spelling.
_lg_tool_tag = "<details class='tool-usage-details' markdown='1'>"
_lg_token_tag = "<details class='token-usage-details' markdown='1'>"
_lg_tool_attrs, _lg_token_attrs = "{.details .tool-usage-details}", "{.details .token-usage-details}"
_lg_tools = re.compile(
    fr"^(?:{_lg_tool_tag}\n*(?:<summary>(?P<summ1>.*?)</summary>\n*)?\n*```json\n+(?P<json1>.*?)\n+```\n+</details>"
    fr"|(?P<fence>:{{3,}}) {re.escape(_lg_tool_attrs)}\n+(?:## (?P<summ2>.*?)\n+)?```json\n+(?P<json2>.*?)\n+```\n+(?P=fence)$)",
    flags=re.DOTALL|re.MULTILINE)
_lg_token = re.compile(
    fr"^(?:{re.escape(_lg_token_tag)}\n*<summary>(?P<tsumm1>.*?)</summary>\n*\n*`(?P<trepr1>.*?)`\n*\n*</details>"
    fr"|(?P<tfence>:{{3,}}) {re.escape(_lg_token_attrs)}\n+## (?P<tsumm2>.*?)\n+`(?P<trepr2>.*?)`\n+(?P=tfence)$)\n?",
    flags=re.DOTALL|re.MULTILINE)

def conv_tools(s):
    "Convert legacy tool/usage envelopes in `s` (both historical spellings) to the fenced JSON wire format. Idempotent."
    def _tool(m):
        tj = m['json1'] if m['json1'] is not None else m['json2']
        try: d = json.loads(tj.strip())
        except Exception: return m[0]
        call = d.get('call') or {}
        res = dict(id=d.get('id'), name=call.get('function'), args=call.get('arguments') or {}, result=d.get('result'))
        if d.get('server'): res['server'] = True
        return fenced(dumps(res, indent=2, ensure_ascii=False), tool_info)
    def _tok(m):
        summ = m['tsumm1'] if m['tsumm1'] is not None else m['tsumm2']
        det = m['trepr1'] if m['trepr1'] is not None else m['trepr2']
        return fenced(dumps(dict(summary=summ, detail=det), ensure_ascii=False), usage_info)
    return _lg_token.sub(_tok, _lg_tools.sub(_tool, s))

In [ ]:
wire_outp = conv_tools(fmt_outp)
test_eq(conv_tools(wire_outp), wire_outp)
assert 'json {.tool}' in wire_outp and 'details' not in wire_outp
Markdown(wire_outp)


I'll solve this step-by-step, using parallel calls where possible.

```json {.tool}
{
  "id": "toolu_01KjnQH2Nsz2viQ7XYpLW3Ta",
  "name": "simple_add",
  "args": {
    "a": 10,
    "b": 5
  },
  "result": "15"
}
```

```json {.tool}
{
  "id": "toolu_01Koi2EZrGZsBbnQ13wuuvzY",
  "name": "simple_add",
  "args": {
    "a": 2,
    "b": 1
  },
  "result": "3"
}
```

Now I need to multiply 15 * 3 before I can do the final division:

```json {.tool}
{
  "id": "toolu_0141NRaWUjmGtwxZjWkyiq6C",
  "name": "multiply",
  "args": {
    "a": 15,
    "b": 3
  },
  "result": "45"
}
```

```json {.usage}
{"summary": "Cache hit: 81.8% | Tokens: total=23,276 input=23,158 (+18,910 cached, 0 new) output=118 (reasoning 23)", "detail": "Usage(prompt_tokens=3, completion_tokens=10, total_tokens=13, raw={'input_tokens': 3, 'cache_creation_input_tokens': 2079, 'cache_read_input_tokens': 2070, 'cache_creation': {'ephemeral_5m_input_tokens': 2079, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 10, 'service_tier': 'standard', 'inference_geo': 'global'})"}
```

`parse_tools` splits the wire form into `(text, tool_dict)` segments - the parsing primitive `fmt2hist` builds on:

In [ ]:
segs = parse_tools(wire_outp)
test_eq(len(segs), 4)
[(txt.strip()[:40], d and d['name']) for txt,d in segs]

[("I'll solve this step-by-step, using para", 'simple_add'),
 ('', 'simple_add'),
 ('Now I need to multiply 15 * 3 before I c', 'multiply'),
 ('```json {.usage}\n{"summary": "Cache hit:', None)]

### Result fences

In [ ]:
#| export
_fence_back = '`````'
_result_re = re.compile(f'\n{_fence_back}result\n(.*?)\n{_fence_back}\n', re.DOTALL)
fence_call_re = re.compile(f'^{_fence_back}(py|bash)\n(.*?)\n{_fence_back}$', re.DOTALL | re.MULTILINE)


In [ ]:
#| export
def extract_fence_call(text):
    "Return (lang, code) if text ends with terminated py/bash fence, else None"
    ms = list(fence_call_re.finditer(text))
    if not ms: return None
    m = ms[-1]
    if not text[m.end():].strip(): return m.group(1), m.group(2)

`fence_call_re` accepts only a complete 5-backtick `py`/`bash` fence on its own lines, and `extract_fence_call` returns the call only when that fence ends the text:

In [ ]:
for bad in ['\n`````py\nprint(1)', '\n```py\nprint(1)\n```\n', 'some text `````py\nprint(1)\n`````\n', '\n`````python\nprint(1)\n`````\n']:
    test_eq(bool(fence_call_re.search(bad)), False)
test_eq(extract_fence_call('prose\n`````bash\nls -la\n`````\n'), ('bash', 'ls -la'))
test_eq(extract_fence_call('\n`````py\nprint(1)\n`````\nmore text'), None)
test_eq(extract_fence_call('hello world'), None)
test_eq(extract_fence_call('\n`````py\nx = 1 | 2\n`````\n'), ('py', 'x = 1 | 2'))
extract_fence_call('\n`````py\nprint(1)\n`````\n')

('py', 'print(1)')

In [ ]:
#| export
def mk_result_fence(output): return f"\n{_fence_back}result\n{output}\n{_fence_back}\n"

def _split_msg_on_fences(msg):
    "Split an assistant Msg on result fences, return list of Msgs"
    if msg.role != 'assistant': return [msg]
    if not _result_re.search(msg.text): return [msg]
    res, asst_parts, tool_parts = [], [], []
    for p in msg.content:
        if   isinstance(p, Thinking): asst_parts.append(p)
        elif isinstance(p, ToolUse):  tool_parts.append(p)
        elif parts := _result_re.split(p.text or ''):
            for i,o in enumerate(parts):
                if not o: continue
                if i % 2 == 0: res.append(Msg(role='assistant', content=asst_parts+[Text(o.strip())]))
                else:          res.append(Msg(role='user',      content=[Text(mk_result_fence(o))]))
    if tool_parts: res.append(Msg(role='assistant', content=tool_parts))
    return res

def split_fence_msgs(msgs):
    "Split all assistant msgs on result fences for wire protocol"
    res = []
    for m in msgs: res.extend(_split_msg_on_fences(m))
    return res

Only an assistant message containing a result fence is split; everything else passes through untouched.

In [ ]:
msg = Msg(role='assistant', content=[Text('Hello world')])
test_eq(_split_msg_on_fences(msg), [msg])
usr = Msg(role='user', content=[Text('`````result\n2\n`````')])
test_eq(_split_msg_on_fences(usr), [usr])
msg


**Msg**

- role: `assistant`

<contents>

**Text** (`text`)

Hello world

::: details

- raw: `None`
- citations: `None`

:::

</contents>

A fence splits one turn into three: the code the model wrote, the result as a `user` message (that's how the wire protocol feeds a result back), and the text that followed.

In [ ]:
msg = Msg(role='assistant', content=[Text('Let me calculate.\n`````py\n1+1\n`````\n\n`````result\n2\n`````\n\nDone.')])
res = _split_msg_on_fences(msg)
test_eq([m.role for m in res], ['assistant', 'user', 'assistant'])
test_eq(['`````py\n1+1' in res[0].text, '`````result\n2\n`````' in res[1].text, 'Done.' in res[2].text], [True]*3)
res

[Msg(role='assistant', content=[Text(raw=None, cache_control=None, text='Let me calculate.\n`````py\n1+1\n`````', citations=None)]),
 Msg(role='user', content=[Text(raw=None, cache_control=None, text='\n`````result\n2\n`````\n', citations=None)]),
 Msg(role='assistant', content=[Text(raw=None, cache_control=None, text='Done.', citations=None)])]

Thinking rides with the assistant text it preceded.

In [ ]:
code_txt = '`````py\nimport random\nprint(random.random())\n`````\n`````result\n42\n`````\n'
msg = Msg(role='assistant', content=[Thinking('The user wants an RNG function...'), Text(code_txt)])
res = split_fence_msgs([msg])
test_eq([[type(p).__name__ for p in m.content] for m in res], [['Thinking','Text'], ['Text']])
res

[Msg(role='assistant', content=[Thinking(raw=None, cache_control=None, text='The user wants an RNG function...', showthink=False), Text(raw=None, cache_control=None, text='`````py\nimport random\nprint(random.random())\n`````', citations=None)]),
 Msg(role='user', content=[Text(raw=None, cache_control=None, text='\n`````result\n42\n`````\n', citations=None)])]

Tool calls are collected into one final assistant message, since a tool call has to be the last thing in its turn.

In [ ]:
msg = Msg(role='assistant', content=[Text(code_txt),
    ToolUse(id='4vy96hyd', name='python', arguments={'code': 'print(random.randint(1, 100))'})])
res = _split_msg_on_fences(msg)
test_eq([m.role for m in res], ['assistant', 'user', 'assistant'])
test_eq([m.content[0].type for m in res], ['text', 'text', 'tool_use'])
res[-1]

**Msg**

- role: `assistant`

<contents>

🔧 **python**(`{'code': 'print(random.randint(1, 100))'}`)

::: details

- id: `4vy96hyd`
- server: `False`
- raw: `None`

:::

</contents>

### fmt2hist

Tool functions can return anything. `tool_text` is the single policy that turns a result into wire text: strings pass through, dicts and lists become JSON (so models see real JSON rather than Python repr, with `default=str` degrading unserializable values field by field), a list of `Part`s (a media tool result) becomes the text parts' text with a `<media>` tag per media part (matching the tag vocabulary clikernel already uses for kernel images), and anything else is `str`'d. fastllm's chat layer and every provider serializer apply it at their boundaries.

The media rendering is deliberately lossy: the block is a text projection, and the media itself only travels inside the live turn. Two lossless designs are known and deferred: serializing the media into the block (rejected for now - base64 balloons stored replies, and display truncation would break the round trip anyway), or a reference scheme where each `<media>` tag carries a `ref` attribute resolved against an out-of-band store, as solveit's attachments already do for images. The tag form is the upgrade seam for both: `_media_tag` grows the attribute and `_extract_tool_parts` learns to resolve it, with no change to the stored format.

In [ ]:
#| export
def _media_tag(p):
    m = getattr(p, 'mime', None)
    return f"<media {p.type} {m}>" if m else f"<media {p.type}>"

def tool_text(
    res, # A tool function's return value
):
    "Canonical string form of a tool result: `Part` lists render as text and `<media>` tags, dicts/lists as JSON, everything else via `str`"
    if isinstance(res, str): return res
    if isinstance(res, list) and all(isinstance(o, Part) for o in res): return '\n'.join(o.ctext for o in res)
    if isinstance(res, (dict, list)): return dumps(res, ensure_ascii=False, default=str)
    return str(res)

In [ ]:
test_eq(tool_text('hi'), 'hi')
test_eq(tool_text({'a': 1, 'ok': True}), '{"a": 1, "ok": true}')
test_eq(tool_text([1, 'x']), '[1, "x"]')
test_eq(tool_text(42), '42')
tool_text({'path': Path('/tmp')})

'{"path": "/tmp"}'

In [ ]:
#| export
def _extract_tool_parts(d:dict):
    "Build (tool_use_part, tool_result_part) from a parsed `{.tool}` block"
    # Skip server tool calls in deserialization (round trip issues with Gemini/Anthropic)
    if not d or d.get('server') or d.get('id') is None: return None
    tu = ToolUse   (id=d['id'], name=d['name'], arguments=d.get('args') or {})
    tr = ToolResult(id=d['id'], name=d['name'], text=tool_text(d.get('result')))
    return tu, tr

In [ ]:
#| export
def fmt2hist(outp:str)->list[Msg]:
    "Transform a formatted output string into fastllm canonical Msgs"
    if usage_info in outp: outp = strip_tools(outp, tools=False)
    if think_start in outp: outp = re_think.sub('', outp)
    if tool_info not in outp:
        msg = Msg(role='assistant', content=[Text(outp.strip() or '.')])
        return _split_msg_on_fences(msg)
    hist, asst_parts, tool_parts = [], [], []
    def flush():
        if tool_parts:
            hist.append(Msg(role='assistant', content=asst_parts.copy()))
            hist.append(Msg(role='tool',      content=tool_parts.copy()))
            asst_parts.clear()
            tool_parts.clear()
    for txt,d in parse_tools(outp.strip()):
        if txt and txt.strip():
            if tool_parts: flush()
            asst_parts.append(Text(txt.strip() or '.'))
        if d and (tp := _extract_tool_parts(d)):
            asst_parts.append(tp[0])
            tool_parts.append(tp[1])
    flush()
    if asst_parts: hist.append(Msg(role='assistant', content=asst_parts))
    if not hist: hist.append(Msg(role='assistant', content=[Text('.')]))
    result = []
    for msg in hist:
        if msg.role == 'assistant': result.extend(_split_msg_on_fences(msg))
        else: result.append(msg)
    if result[-1].role == 'tool': result.append(Msg(role='assistant', content=[Text('.')]))
    return result

See how we can turn that one formatted output string back into a list of Msg:

In [ ]:
h = fmt2hist(wire_outp)
test_eq([m.role for m in h], ['assistant','tool','assistant','tool','assistant'])
h[1]

**Msg**

- role: `tool`

<contents>

**ToolResult** (`tool_result`)

15

::: details

- raw: `None`
- id: `toolu_01KjnQH2Nsz2viQ7XYpLW3Ta`
- name: `simple_add`
- arguments: `{}`
- server: `False`

:::

**ToolResult** (`tool_result`)

3

::: details

- raw: `None`
- id: `toolu_01Koi2EZrGZsBbnQ13wuuvzY`
- name: `simple_add`
- arguments: `{}`
- server: `False`

:::

</contents>

A tool response can be a string or a list of tool blocks (e.g., an image url block). To allow users to specify if a response should not be immediately stringified, we provide the `ToolResponse` datatype users can wrap their return statement in.

In [ ]:
#| export
class ToolResponse(BasicRepr):
    def __init__(self, content): store_attr()  # list of (text, result) pairs
    def __eq__(self, o): return type(o) is type(self) and self.__dict__ == o.__dict__
    def __hash__(self): return hash(str(self.content))

`StopResponse` and `FullResponse` are `str` subclasses that mark a string's handling downstream: a `StopResponse` tool result ends a tool loop, and a `FullResponse` must never be truncated. `_trunc_str` honors the latter (along with fastcore's `Safe` and `PrettyString`, checked by class name so no import is needed), and the `𝍁...𝍁` marker is the same contract for strings that crossed a serialization boundary:

In [ ]:
#| export
class StopResponse(str): pass
class FullResponse(str): pass

In [ ]:
#| export
def trunc_str(s, mx=2000, skip=10, replace="TRUNCATED"):
    "Truncate `s` to `mx` chars max, adding `replace` if truncated; `mx=None` disables truncation"
    if mx is None or isinstance_str(s, ('FullResponse','Safe','PrettyString')): return s
    if not isinstance(s, str): s = str(s)
    s = type(s)(s.rstrip())
    if len(s)>2 and s[0]=='𝍁' and s[-1]=='𝍁':
        s = s[1:-1]
        if replace: return s
    if mx is None or len(s)<=mx: return s
    s = s[skip:mx-skip]
    ss = s.split(' ')
    if len(ss[-1])>150: ss[-1] = ss[-1][:5]
    s = ' '.join(ss)
    if skip: s = f"…{s}"
    s = f"{s}…"
    if replace: s = f"<{replace}>{s}</{replace}>"
    return s

In [ ]:
test_eq(trunc_str('𝍁xxxxxxxxxx𝍁', mx=5), 'xxxxxxxxxx')
test_eq(trunc_str(Safe('xxxxxxxxxx'), mx=5), 'xxxxxxxxxx')
test_eq(trunc_str(FullResponse('xxxxxxxxxx'), mx=5), 'xxxxxxxxxx')
test_eq(trunc_str('xxxxxxxxxx', mx=5, skip=0), '<TRUNCATED>xxxxx…</TRUNCATED>')
test_eq(trunc_str('xxxxxxxxxx', mx=5, skip=1), '<TRUNCATED>…xxx…</TRUNCATED>')
test_eq(trunc_str('xxxxxxxxxx', mx=None), 'xxxxxxxxxx')

In [ ]:
#| export
def _trunc_param(v, mx=40):
    "Truncate and escape param value for display"
    tp = trunc_str(str(v).replace('`', r'\`'), mx=mx, replace=None, skip=0)
    try: return dumps(tp, ensure_ascii=False)
    except Exception: return repr(tp).replace('\\\\', '\\')

def _tc_summary(tr):
    "Format tool call as a `func(params)→result` code span"
    params = ', '.join(f"{k}={_trunc_param(v)}" for k,v in tr.arguments.items())
    res = f"→{_trunc_param(tr.text)}" if tr.text else ''
    txt = f"{tr.name}({params}){res}"
    ticks = '`'*(max(map(len, re.findall('`+', txt)), default=0)+1)
    pad = ' ' if '`' in txt else ''
    return f"{ticks}{pad}{txt}{pad}{ticks}"

In [ ]:
#| export
def mk_tr_details(tr, mx=2000):
    "Create the `{.tool}` wire block for a tool call; `mx=None` disables truncation"
    args = {k:trunc_str(v, mx=None if mx is None else mx*5) if isinstance(v, str) else v for k,v in tr.arguments.items()}
    res = dict(id=tr.id, name=tr.name, args=args, result=trunc_str(tool_text(tr.text), mx=mx))
    if tr.server: res['server'] = True
    return "\n\n" + fenced(dumps(res, indent=2, ensure_ascii=False), tool_info) + "\n\n"

Every part has two display forms, and each is a method on the part itself, so nothing that renders needs to know what kinds of part exist.

`formatted` is the *stream* form, for renderers that append each item as it arrives: text as its raw fragment, a thought as a 🧠 glyph (or the thought itself when the stream stamped `showthink`), a tool call as a pending ⏳ row until its result lands.

`doc(showthink=False, mx=2000)` is the *document* form, for rendering finished messages: text stripped, a thought as a `<details>` block only when asked for, a tool call and its result as the `{.tool}` wire block. `hist2fmt` below is little more than a walk over messages calling it.

In [ ]:
#| export
@patch(as_prop=True)
def formatted(self:Part): return self.text or ''
@patch
def doc(self:Part, showthink=False, mx=2000): return (self.text or '').strip()

@patch(as_prop=True)
def formatted(self:Thinking): return (self.text or '') if self.showthink else '🧠'
@patch
def doc(self:Thinking, showthink=False, mx=2000):
    if not (showthink and self.text): return ''
    return f'{think_start}\n::: details\n\n## Thinking\n\n{self.text.strip()}\n\n:::\n{think_end}'

@patch(as_prop=True)
def formatted(self:ToolUse): return '' if self.server else f"\n- ⏳ {_tc_summary(self)} ⏳\n"
@patch
def doc(self:ToolUse, showthink=False, mx=2000):
    "A server call's result is implicit, so it renders as a completed block; any other pending call is an ⏳ row"
    if not self.server: return self.formatted.strip()
    return mk_tr_details(self.replace(text='Server tool call executed.'), mx=mx).strip()

@patch(as_prop=True)
def formatted(self:ToolResult): return mk_tr_details(self)
@patch
def doc(self:ToolResult, showthink=False, mx=2000):
    "A result with no id can't be re-parsed into history (e.g. Gemini code execution), so it doesn't render"
    return mk_tr_details(self, mx=mx).strip() if self.id else ''

`ctext` is a part's compact content form, used when it appears inside a tool result. By default it *is* `formatted` -- one rendering identity per class -- and `Media` is the exception, since its `formatted` would be the full data URL:

In [ ]:
#| export
@patch(as_prop=True)
def ctext(self:Part): return self.formatted

@patch(as_prop=True)
def ctext(self:Media): return _media_tag(self)

In [ ]:
test_eq(Text('caption').ctext, 'caption')
test_eq(Thinking('secret').ctext, '🧠')
test_eq(tool_text([Text('caption'), InputImage('data:image/png;base64,xxx', mime='image/png')]), 'caption\n<media input_image image/png>')
test_eq(tool_text([InputFile('https://x.co/a.pdf')]), '<media input_file>')

A media tool result (`text` holding `Part`s) renders through `tool_text`, so the block stays a readable text projection: captions verbatim, each media part as its `<media>` tag. On re-parse the model sees that placeholder text - the media itself only travels inside the live turn, via the provider wire converters.

In [ ]:
mtr = ToolResult(id='t1', name='screenshot', text=[Text('The login page.'), InputImage('data:image/png;base64,iVBORw0K', mime='image/png')])
assert '<media input_image image/png>' in mk_tr_details(mtr)
Markdown(mk_tr_details(mtr))



```json {.tool}
{
  "id": "t1",
  "name": "screenshot",
  "args": {},
  "result": "The login page.\n<media input_image image/png>"
}
```



In [ ]:
tc = ToolUse(id='tc1', name='simple_add', arguments={'a':3,'b':5})
test_eq(tc.formatted, '\n- ⏳ `simple_add(a="3", b="5")` ⏳\n')
test_eq(ToolUse(id='s1', name='web_search', server=True).formatted, '')
test_eq(Thinking('deep thought').formatted, '🧠')
test_eq(Thinking('deep thought', showthink=True).formatted, 'deep thought')
test_eq(Text('hi').formatted, 'hi')
tc.formatted

'\n- ⏳ `simple_add(a="3", b="5")` ⏳\n'

`hist2fmt` is the inverse: it renders assistant/tool messages back into one formatted output string, with each tool call as the `<details>` block `fmt2hist` parses. This is how a captured conversation becomes an editable reply (e.g. a Solveit prompt output, or an llmsurgery dialog): text stays text, and every `tool_use`/`tool_result` pair folds into a details block carrying the call and its result. Results longer than `mx` are truncated by `mk_tr_details`, so the roundtrip is exact only within that limit; pass `mx=None` for an exact roundtrip with no truncation.

In [ ]:
#| export
def hist2fmt(msgs:list[Msg], mx=2000, showthink=False)->str:
    "Render assistant/tool `msgs` as one formatted output string, the inverse of `fmt2hist`"
    tus, out = {}, []
    for m in msgs:
        if m.role == 'assistant':
            for p in m.content:
                if isinstance(p, ToolUse) and not p.server: tus[p.id] = p
                else: out.append(p.doc(showthink=showthink, mx=mx))
        elif m.role == 'tool':
            for p in m.content:
                if not isinstance(p, ToolResult): continue
                tu = tus.pop(p.id, None)                       # results don't carry the call's arguments
                out.append(p.replace(arguments=tu.arguments if tu else {}).doc(mx=mx))
        else: raise ValueError(f"hist2fmt renders assistant and tool messages only, got {m.role!r}")
    out += [p.doc() for p in tus.values()]                     # calls still awaiting a result
    return '\n\n'.join(o for o in out if o)

In [ ]:
Markdown(hist2fmt(fmt2hist(wire_outp)))

I'll solve this step-by-step, using parallel calls where possible.

```json {.tool}
{
  "id": "toolu_01KjnQH2Nsz2viQ7XYpLW3Ta",
  "name": "simple_add",
  "args": {
    "a": 10,
    "b": 5
  },
  "result": "15"
}
```

```json {.tool}
{
  "id": "toolu_01Koi2EZrGZsBbnQ13wuuvzY",
  "name": "simple_add",
  "args": {
    "a": 2,
    "b": 1
  },
  "result": "3"
}
```

Now I need to multiply 15 * 3 before I can do the final division:

```json {.tool}
{
  "id": "toolu_0141NRaWUjmGtwxZjWkyiq6C",
  "name": "multiply",
  "args": {
    "a": 15,
    "b": 3
  },
  "result": "45"
}
```

.

Parsing that rendering gives back exactly the messages we started from - `fmt2hist` and `hist2fmt` are inverses on the wire form.

In [ ]:
h2 = fmt2hist(hist2fmt(h))
test_eq(h2, h)

A call with no result yet renders as a pending row, and on re-parse that row is plain text, so an unmatched `tool_use` can never re-enter history:

In [ ]:
pend = Msg('assistant', [Text('Calling...'), ToolUse(id='p1', name='f', arguments={'x':1})])
s = hist2fmt([pend])
test_eq(s, 'Calling...\n\n- ⏳ `f(x="1")` ⏳')
test_eq(fmt2hist(s)[0].content[0].text, s)
s

'Calling...\n\n- ⏳ `f(x="1")` ⏳'

Thinking renders only on request, and even then its block strips on re-parse, so thinking never re-enters history:

In [ ]:
tm = Msg('assistant', [Thinking('hmm, sums'), Text('The answer is 8.')])
test_eq(hist2fmt([tm]), 'The answer is 8.')
test_eq('hmm, sums' in hist2fmt([tm], showthink=True), True)
test_eq(fmt2hist(hist2fmt([tm], showthink=True))[0].content[0].text, 'The answer is 8.')
Markdown(hist2fmt([tm], showthink=True))

<!--think_start-->
::: details

## Thinking

hmm, sums

:::
<!--think_end-->

The answer is 8.

A `ToolResult` with no `id` (Gemini code execution reports results this way) can't be paired with a call on re-parse, so it doesn't render at all - the round trip stays exact.

In [ ]:
gm = Msg('assistant', [Text('Ran it.'), ToolResult(text='42', raw={'codeExecutionResult': {}})])
test_eq(hist2fmt([gm]), 'Ran it.')
test_eq(fmt2hist(hist2fmt([gm])), [Msg('assistant', [Text('Ran it.')])])

## Building conversations

`mk_msg` turns any convenient content form -- a string, bytes, a list of mixed content, a role/content dict, a `Msg`, or a `Completion` -- into a canonical `Msg`:

In [ ]:
#| export
def mk_msg(
    content,      # Content: str, bytes (image), list of mixed content, or dict w 'role' and 'content' fields
    role="user"    # Message role if content isn't already a dict/Message
):
    "Create a LiteLLM compatible message."
    if content is None: return None
    if isinstance(content, Msg): return content
    if isinstance(content, Completion): return content.message
    if isinstance(content, list) and len(content) == 1 and isinstance(content[0], str): parts = [Text(content[0])]
    elif isinstance(content, list): parts = [mk_content(o) for o in content]
    elif isinstance(content, dict): return Msg(role=content['role'], content=[Text(content['content'])])
    else: parts = [Text(content)]
    return Msg(role=role, content=parts)

In [ ]:
m = mk_msg('hey')
test_eq(m, mk_msg(['hey']))
m

**Msg**

- role: `user`

<contents>

**Text** (`text`)

hey

::: details

- raw: `None`
- citations: `None`

:::

</contents>

Now lets make it easy to provide entire conversations:

In [ ]:
#| export
def mk_msgs(
    msgs    # List of messages (each: str, bytes, list, Msg, or Completion)
):
    "Create a list of fastllm canonical Msgs."
    if not msgs: return []
    if not isinstance(msgs, list): msgs = [msgs]
    msgs = L(msgs).map(lambda m: fmt2hist(m) if isinstance(m,str) and (tool_info in m or usage_info in m) else [m]).concat()
    res, role = [], 'user'
    for m in msgs:
        res.append(msg := mk_msg(m, role=role))
        role = 'assistant' if msg.role in ('user','tool') else 'user'
    return res

With `mk_msgs` you can easily provide a whole conversation:

In [ ]:
msgs = mk_msgs(['Hey!',"Hi there!","How are you?","I'm doing fine and you?"])
msgs

[Msg(role='user', content=[Text(raw=None, cache_control=None, text='Hey!', citations=None)]),
 Msg(role='assistant', content=[Text(raw=None, cache_control=None, text='Hi there!', citations=None)]),
 Msg(role='user', content=[Text(raw=None, cache_control=None, text='How are you?', citations=None)]),
 Msg(role='assistant', content=[Text(raw=None, cache_control=None, text="I'm doing fine and you?", citations=None)])]

In [ ]:
msgs[-2]

**Msg**

- role: `user`

<contents>

**Text** (`text`)

How are you?

::: details

- raw: `None`
- citations: `None`

:::

</contents>

Who's speaking at when is automatically inferred.
Even when there are multiple tools being called in parallel (which LiteLLM supports!).

In [ ]:
msgs = mk_msgs(['Tell me the weather in Paris and Rome',
    'Assistant calls weather tool two times',
    {'role':'tool','content':'Weather in Paris is ...'},
    {'role':'tool','content':'Weather in Rome is ...'},
    'Assistant returns weather',
    'Thanks!'])
msgs

[Msg(role='user', content=[Text(raw=None, cache_control=None, text='Tell me the weather in Paris and Rome', citations=None)]),
 Msg(role='assistant', content=[Text(raw=None, cache_control=None, text='Assistant calls weather tool two times', citations=None)]),
 Msg(role='tool', content=[Text(raw=None, cache_control=None, text='Weather in Paris is ...', citations=None)]),
 Msg(role='tool', content=[Text(raw=None, cache_control=None, text='Weather in Rome is ...', citations=None)]),
 Msg(role='assistant', content=[Text(raw=None, cache_control=None, text='Assistant returns weather', citations=None)]),
 Msg(role='user', content=[Text(raw=None, cache_control=None, text='Thanks!', citations=None)])]

In [ ]:
#| hide
test_eq([m.role for m in msgs],['user','assistant','tool','tool','assistant','user'])

For ease of use, if `msgs` is not already in a `list`, it will automatically be wrapped inside one. This way you can pass a single prompt into `mk_msgs` and get back a ~~LiteLLM~~ fastllm compatible msg history.

In [ ]:
msgs = mk_msgs("Hey")
msgs

[Msg(role='user', content=[Text(raw=None, cache_control=None, text='Hey', citations=None)])]

In [ ]:
#| hide
msgs = mk_msgs({'role':'tool','content':'fake tool result'})
msgs

[Msg(role='tool', content=[Text(raw=None, cache_control=None, text='fake tool result', citations=None)])]

In [ ]:
msgs = mk_msgs(['Hey!',"Hi there!","How are you?","I'm fine, you?"])
msgs

[Msg(role='user', content=[Text(raw=None, cache_control=None, text='Hey!', citations=None)]),
 Msg(role='assistant', content=[Text(raw=None, cache_control=None, text='Hi there!', citations=None)]),
 Msg(role='user', content=[Text(raw=None, cache_control=None, text='How are you?', citations=None)]),
 Msg(role='assistant', content=[Text(raw=None, cache_control=None, text="I'm fine, you?", citations=None)])]

However, beware that if you use `mk_msgs` for a single message, consisting of multiple parts.
Then you should be explicit, and make sure to wrap those multiple messages in two lists:

1. One list to show that they belong together in one message (the inner list).
2. Another, because mk_msgs expects a list of multiple messages (the outer list).

This is common when pairing a question with an image, for example -- here shown with two text parts:

In [ ]:
msgs = mk_msgs([['Whats in this image?', 'https://example.com/puppy.jpg']])
test_eq(len(msgs), 1)
test_eq(len(msgs[0].content), 2)
msgs[0]

**Msg**

- role: `user`

<contents>

**Text** (`text`)

Whats in this image?

::: details

- raw: `None`
- citations: `None`

:::

**Text** (`text`)

https://example.com/puppy.jpg

::: details

- raw: `None`
- citations: `None`

:::

</contents>

In [ ]:
#| hide
#| eval: false
import nbdev; nbdev.nbdev_export()